Exercise W2D2.1 — Dataset health check

Load supply_chain.csv and write a complete dataset health check that prints:

— Shape, column names, and dtypes
— Count and percentage of missing values per column
— Summary statistics for quantity and unit_price
— Number of unique products and regions
— The distribution of orders by region (count and % share)
— The date range of the data

In [4]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("supply_chain.csv", parse_dates=["date"])

print("=== DATASET HEALTH CHECK ===")
print(f"\nShape:   {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

print("\n--- Column types ---")
print(df.dtypes)

print("\n--- Missing values ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
miss_df = pd.DataFrame({"count": missing, "pct": missing_pct})
print(miss_df)


print("\n--- Numeric summary ---")
print(df[["quantity", "unit_price"]].describe().round(2))

print("\n--- Categorical ---")
print(f"Unique products: {df['product'].nunique()}")
print(f"Unique regions:  {df['region'].nunique()}")


print("\n--- Region distribution ---")
region_dist = pd.DataFrame({
    "count": df["region"].value_counts(),
    "pct":   df["region"].value_counts(normalize=True).mul(100).round(1)
})
print(region_dist)

print(f"\n--- Date range ---")
print(f"From: {df['date'].min().date()}")
print(f"To:   {df['date'].max().date()}")

=== DATASET HEALTH CHECK ===

Shape:   200 rows × 6 columns
Columns: ['date', 'product', 'category', 'quantity', 'unit_price', 'region']

--- Column types ---
date          datetime64[ns]
product               object
category              object
quantity               int64
unit_price           float64
region                object
dtype: object

--- Missing values ---
            count  pct
date            0  0.0
product         0  0.0
category        0  0.0
quantity        0  0.0
unit_price      0  0.0
region          0  0.0

--- Numeric summary ---
       quantity  unit_price
count    200.00      200.00
mean     262.45       52.49
std      142.26       55.45
min        6.00        4.75
25%      147.00       14.27
50%      261.50       21.76
75%      389.25       92.08
max      494.00      157.35

--- Categorical ---
Unique products: 5
Unique regions:  4

--- Region distribution ---
        count   pct
region             
North      61  30.5
South      53  26.5
West       43  21.5
Eas

Exercise W2D2.2 — Revenue analysis with computed columns

Starting from supply_chain.csv, build an enriched DataFrame by adding these computed columns:

— revenue (quantity × unit_price)
— month (integer 1–12) and month_name extracted from date
— rev_band: "low" (<1000), "medium" (1000–5000), "high" (>5000)
— discount_eligible: True if quantity > 200

Then answer using Pandas operations — no loops:
— How many orders are in each rev_band?
— What percentage of orders are discount eligible?
— What is the total revenue for high rev_band orders?

In [5]:
import pandas as pd

df = pd.read_csv("supply_chain.csv", parse_dates=["date"])

# Computed columns
df["revenue"]           = (df["quantity"] * df["unit_price"]).round(2)
df["month"]             = df["date"].dt.month
df["month_name"]        = df["date"].dt.month_name()
df["discount_eligible"] = df["quantity"] > 200
df["rev_band"]          = df["revenue"].apply(
    lambda x: "high" if x > 5000 else "medium" if x >= 1000 else "low"
)

print(df[["product", "revenue", "rev_band", "discount_eligible"]].head(5))

print("\n--- Orders per rev_band ---")
print(df["rev_band"].value_counts())

disc_pct = df["discount_eligible"].mean() * 100
print(f"\nDiscount eligible: {disc_pct:.1f}% of orders")

high_total = df.loc[df["rev_band"] == "high", "revenue"].sum()
print(f"High band total revenue: £{high_total:,.2f}")

     product   revenue rev_band  discount_eligible
0  Connector    519.18      low              False
1   Sensor Y  17923.46     high               True
2   Gadget X  69625.86     high               True
3   Sensor Y  37015.23     high               True
4   Sensor Y  22638.32     high               True

--- Orders per rev_band ---
rev_band
high      108
medium     57
low        35
Name: count, dtype: int64

Discount eligible: 60.0% of orders
High band total revenue: £2,857,902.57


Exercise W2D2.3 — Multi-condition order report

Build a filtered order report that a sales manager could actually use. Starting from the enriched DataFrame (with revenue and rev_band added), produce three separate filtered DataFrames and print a clean summary for each:

1. Priority orders: Electronics category, revenue > 5000, any region
2. Underperforming orders: quantity < 50 AND revenue < 500, sorted by revenue ascending
3. Regional spotlight: all Widget A orders from North or East regions, sorted by date

For each group print: row count, total revenue, average quantity, and the top 5 rows showing product, region, quantity, and revenue.

In [6]:
import pandas as pd

df = pd.read_csv("supply_chain.csv", parse_dates=["date"])
df["revenue"] = (df["quantity"] * df["unit_price"]).round(2)
df["rev_band"] = df["revenue"].apply(
    lambda x: "high" if x > 5000 else "medium" if x >= 1000 else "low"
)

def print_group(title, subset):
    print(f"\n{'='*50}")
    print(f"  {title}")
    print(f"{'='*50}")
    print(f"  Orders:    {subset.shape[0]}")
    print(f"  Revenue:   £{subset['revenue'].sum():,.2f}")
    print(f"  Avg qty:   {subset['quantity'].mean():.1f}")
    print(f"\n  Top 5 rows:")
    print(subset[["product", "region", "quantity", "revenue"]].head(5).to_string(index=False))

# 1. Priority orders
priority = df[(df["category"] == "Electronics") & (df["revenue"] > 5000)]
print_group("1. Priority Orders — Electronics & Revenue > £5000", priority)

# 2. Underperforming
underperf = df[(df["quantity"] < 50) & (df["revenue"] < 500)].sort_values("revenue")
print_group("2. Underperforming — qty < 50 and revenue < £500", underperf)

# 3. Widget A North/East
widget_ne = df[(df["product"] == "Widget A") &df["region"].isin(["North", "East"])].sort_values("date")
print_group("3. Widget A — North & East regions by date", widget_ne)


  1. Priority Orders — Electronics & Revenue > £5000
  Orders:    71
  Revenue:   £2,593,897.05
  Avg qty:   295.5

  Top 5 rows:
 product region  quantity  revenue
Sensor Y  North       202 17923.46
Gadget X  South       474 69625.86
Sensor Y  North       399 37015.23
Sensor Y  South       244 22638.32
Gadget X   West       101 15775.19

  2. Underperforming — qty < 50 and revenue < £500
  Orders:    9
  Revenue:   £2,082.81
  Avg qty:   24.4

  Top 5 rows:
  product region  quantity  revenue
Connector  South         6    28.92
Connector  North        21   109.41
 Widget A  North        10   153.70
Connector  South        39   186.81
Connector   West        41   204.59

  3. Widget A — North & East regions by date
  Orders:    21
  Revenue:   £90,462.80
  Avg qty:   290.3

  Top 5 rows:
 product region  quantity  revenue
Widget A  North       407  5816.03
Widget A  North       356  5087.24
Widget A  North       438  6876.60
Widget A  North       165  2524.50
Widget A  North       424